In [0]:
%sql

select * from bronze_dev.external_tables.client_brz_ext

In [0]:
-- Insert birth_date values greater than 1969 for all rows in the table
UPDATE bronze_dev.external_tables.client_brz_ext
SET birth_date = date_add('1969-01-01', 1)
WHERE birth_date IS NULL OR birth_date <= '1969-01-01'

In [0]:
truncate table bronze_dev.external_tables.client_brz_ext

In [0]:
select * from bronze_dev.external_tables.client_brz_ext version as of 2

In [0]:
RESTORE TABLE bronze_dev.external_tables.client_brz_ext TO VERSION AS OF 2

In [0]:
%python
file = dbutils.fs.ls("s3://lakehouse-rodrigo-reyes/bronze-dev/external_tables/client_brz_ext")
display(file)

In [0]:
%python
from pyspark.sql.functions import current_timestamp, date_format, current_date

In [0]:
Describe history bronze_dev.external_tables.client_brz_ext

In [0]:
%python
from pyspark.sql.functions import current_timestamp, current_date

In [0]:
%python
clients_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("delimiter", ",")
    .load("s3://lakehouse-rodrigo-reyes/landing_dev/*.csv")
).withColumn("ingest_timestamp", current_timestamp()).withColumn("ingest_date", current_date())

clients_df.display()

In [0]:
%python
# Ajustar el DataFrame para que coincida con el esquema de la tabla Delta
clients_df_fixed = clients_df \
    .withColumnRenamed("ID_Cliente", "id") \
    .withColumnRenamed("Nombre_Completo", "name") \
    .withColumnRenamed("Email", "email") \
    .withColumnRenamed("Ciudad", "country") \
    .withColumnRenamed("Saldo_Pendiente", "role") \
    .withColumnRenamed("Es_Activo", "last_name")

# Seleccionar solo las columnas requeridas por la tabla (sin birth_date)
clients_df_fixed = clients_df_fixed.select(
    "id", "name", "last_name", "email", "country", "role", "ingest_timestamp", "ingest_date"
)

clients_df_fixed.writeTo('bronze_dev.external_tables.client_brz_ext').append()